In [2]:
import pandas as pd
import time
from geopy.geocoders import Nominatim

# Load CSV files
df_konvensional = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_konvensional.csv')
df_syariah = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_syariah.csv')

print("Data Konvensional shape:", df_konvensional.shape)
print("Data Syariah shape:", df_syariah.shape)
print("\nKolom:", df_konvensional.columns.tolist())

Data Konvensional shape: (1863, 4)
Data Syariah shape: (194, 4)

Kolom: ['Provinsi', 'Kabupaten/Kota', 'Nama Bank', 'Kode Bank']


In [3]:
import pandas as pd
from geopy.geocoders import Nominatim
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# Load CSV files
print('Loading CSV files...')
df_konvensional = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_konvensional.csv')
df_syariah = pd.read_csv('/Users/mraffyzeidan/Learning/xcap/Client-BPRSK/BPRSK/Backup/ojk_cfs_bpr_syariah.csv')

print(f"Data Konvensional: {df_konvensional.shape}")
print(f"Data Syariah: {df_syariah.shape}")

# Extract unique banks
banks_konvensional = df_konvensional[['Nama Bank', 'Provinsi', 'Kabupaten/Kota']].drop_duplicates()
banks_syariah = df_syariah[['Nama Bank', 'Provinsi', 'Kabupaten/Kota']].drop_duplicates()

all_banks = pd.concat([
    banks_konvensional.assign(Jenis='Konvensional'),
    banks_syariah.assign(Jenis='Syariah')
], ignore_index=True)

print(f"Total bank konvensional unik: {len(banks_konvensional)}")
print(f"Total bank syariah unik: {len(banks_syariah)}")
print(f"Total bank unik: {len(all_banks)}")

Loading CSV files...
Data Konvensional: (1863, 4)
Data Syariah: (194, 4)
Total bank konvensional unik: 1861
Total bank syariah unik: 194
Total bank unik: 2055


In [4]:
all_banks

,Nama Bank,Provinsi,Kabupaten/Kota,Jenis
0,PT Bank Perekonomian Rakyat Cikarang Raharja,Provinsi Jawa Barat,Kab. Bekasi,Konvensional
1,PT Bank Perekonomian Rakyat Dana Multi Guna,Provinsi Jawa Barat,Kab. Bekasi,Konvensional
2,PT. BPR Siwa Raharja Utama,Provinsi Jawa Barat,Kab. Bekasi,Konvensional
3,PT Bank Perekonomian Rakyat Binadana Makmur,Provinsi Jawa Barat,Kab. Bekasi,Konvensional
4,PT Bank Perekonomian Rakyat Sentral Mandiri,Provinsi Jawa Barat,Kab. Bekasi,Konvensional
...,...,...,...,...
2050,PT Bank Perekonomian Rakyat Syariah Fajar Seja...,Provinsi Bali,Kab. Badung,Syariah
2051,PT. BPRS Muamalat Yotefa,Provinsi Papua,Kab. Jayapura,Syariah
2052,PT Bank Perekonomian Rakyat Syariah Saruma Sej...,Provinsi Maluku Utara,Kab. Halmahera Selatan,Syariah
2053,PT Bank Perekonomian Rakyat Syariah Bahari Ber...,Provinsi Maluku Utara,Kota Ternate,Syariah


In [ ]:
# GEOCODING PAKE SELENIUM + GOOGLE MAPS (PARALLEL + HEADLESS)
# pip install selenium

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import re
import random
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed

# Random User Agents
USER_AGENTS = [
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:121.0) Gecko/20100101 Firefox/121.0',
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:121.0) Gecko/20100101 Firefox/121.0',
]

def extract_coordinates_from_url(url):
    """Extract lat, lon dari Google Maps URL"""
    # Pattern: @-6.234900,106.989600
    pattern = r'@(-?\d+\.\d+),(-?\d+\.\d+)'
    match = re.search(pattern, url)
    if match:
        return float(match.group(1)), float(match.group(2))
    return None, None

def create_driver():
    """Create new Firefox driver instance"""
    firefox_options = Options()
    
    # HEADLESS MODE (ga muncul browser window)
    firefox_options.add_argument('--headless')
    
    # PRIVATE BROWSING (Incognito)
    firefox_options.add_argument('-private')
    
    # ANTI-DETECTION
    firefox_options.set_preference("dom.webdriver.enabled", False)
    firefox_options.set_preference('useAutomationExtension', False)
    firefox_options.set_preference("general.useragent.override", random.choice(USER_AGENTS))
    
    # PRIVACY & SECURITY
    firefox_options.set_preference("privacy.trackingprotection.enabled", True)
    firefox_options.set_preference("privacy.donottrackheader.enabled", True)
    firefox_options.set_preference("privacy.firstparty.isolate", True)
    firefox_options.set_preference("network.cookie.cookieBehavior", 1)
    firefox_options.set_preference("network.http.referer.spoofSource", True)
    
    # DISABLE NOTIFICATIONS & GEOLOCATION
    firefox_options.set_preference("dom.webnotifications.enabled", False)
    firefox_options.set_preference("geo.enabled", False)
    firefox_options.set_preference("geo.provider.use_corelocation", False)
    
    # FASTER LOADING (disable images)
    firefox_options.set_preference("permissions.default.image", 2)
    
    # DISABLE CACHE
    firefox_options.set_preference("browser.cache.disk.enable", False)
    firefox_options.set_preference("browser.cache.memory.enable", False)
    firefox_options.set_preference("browser.cache.offline.enable", False)
    
    # Setup geckodriver path
    service = Service(executable_path='/opt/homebrew/bin/geckodriver')
    
    return webdriver.Firefox(service=service, options=firefox_options)

def geocode_google_maps_selenium(bank_name, city, province):
    """Geocode dengan Selenium + Google Maps"""
    driver = None
    try:
        # Create driver
        driver = create_driver()
        wait = WebDriverWait(driver, 15)
        
        # Build query
        query = f"{bank_name}, {city}, {province}, Indonesia"
        
        # Go to Google Maps
        driver.get("https://www.google.com/maps")
        time.sleep(2)
        
        # Find search box dan search
        search_box = wait.until(EC.presence_of_element_located((By.XPATH, '//*[@id="searchboxinput"]')))
        search_box.clear()
        search_box.send_keys(query)
        search_box.send_keys(Keys.RETURN)
        
        # Wait for results to load
        time.sleep(3)
        
        # Try to click first result (div dengan class yang contains result)
        try:
            # Cari first result link - biasanya ada di div dengan attribute aria-label
            first_result = wait.until(EC.element_to_be_clickable((
                By.CSS_SELECTOR, 
                'a[href*="@"]'  # Link yang ada koordinat di href
            )))
            first_result.click()
            time.sleep(2)
        except (TimeoutException, NoSuchElementException):
            # Kalo ga bisa click, langsung extract dari URL
            pass
        
        # Extract coordinates dari URL
        current_url = driver.current_url
        lat, lon = extract_coordinates_from_url(current_url)
        
        if lat and lon:
            # Validasi Indonesia
            if -15 <= lat <= 10 and 90 <= lon <= 150:
                return lat, lon, "SUCCESS"
        
    except Exception as e:
        return None, None, "FAILED"
    
    finally:
        if driver:
            driver.quit()
    
    return None, None, "FAILED"

print("\nGEOCODING DENGAN SELENIUM + GOOGLE MAPS (PARALLEL + HEADLESS)")

all_banks['Latitude'] = None
all_banks['Longitude'] = None

berhasil = 0
gagal_count = 0
processed = 0
results = {}

# Parallel processing dengan ThreadPoolExecutor
max_workers = 3  # 3 browsers parallel

try:
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        futures = {}
        for idx, row in all_banks.iterrows():
            future = executor.submit(
                geocode_google_maps_selenium,
                row['Nama Bank'],
                row['Kabupaten/Kota'],
                row['Provinsi']
            )
            futures[future] = idx
        
        # Process completed tasks
        for future in tqdm(as_completed(futures), total=len(futures)):
            idx = futures[future]
            lat, lon, status = future.result()
            
            results[idx] = (lat, lon, status)
            processed += 1
            
            row = all_banks.iloc[idx]
            
            if status == "SUCCESS":
                berhasil += 1
                if berhasil <= 30:
                    print(f"✓ [{berhasil}] {row['Nama Bank'][:40]} | ({lat:.6f}, {lon:.6f})")
            else:
                gagal_count += 1
                if gagal_count <= 20:
                    print(f"✗ [{gagal_count}] {row['Nama Bank'][:40]}")
            
            # Auto-save tiap 50 bank
            if processed % 50 == 0:
                for i, (lat, lon, status) in results.items():
                    all_banks.at[i, 'Latitude'] = lat
                    all_banks.at[i, 'Longitude'] = lon
                all_banks.to_csv('BankLongLat_PROGRESS.csv', index=False)
                print(f"\n SAVED: {processed}/{len(all_banks)} | Success: {berhasil} | Fail: {gagal_count}\n")

except KeyboardInterrupt:
    print("\n STOPPED!")

# Final save
for idx, (lat, lon, status) in sorted(results.items()):
    all_banks.at[idx, 'Latitude'] = lat
    all_banks.at[idx, 'Longitude'] = lon

all_banks.to_csv('BankLongLat_PROGRESS.csv', index=False)

print(f"\n{'='*70}")
print(f"HASIL GEOCODING SELENIUM + GOOGLE MAPS:")
print(f"{'='*70}")
print(f"Total: {len(all_banks)}")
print(f"Processed: {processed}/{len(all_banks)} ({processed/len(all_banks)*100:.1f}%)")
print(f"✓ Success: {berhasil} ({berhasil/len(all_banks)*100:.1f}%)")
print(f"✗ Failed: {gagal_count} ({gagal_count/len(all_banks)*100:.1f}%)")
print(f"Success Rate: {berhasil/processed*100:.1f}%")
print(f"{'='*70}")
print(f"\n Saved to: BankLongLat_PROGRESS.csv")


🗺️ GEOCODING DENGAN SELENIUM + GOOGLE MAPS (PARALLEL + HEADLESS)
⚠️ Headless mode - browser jalan di background (ga kelihatan)
⚠️ 3 parallel workers
⚠️ Random User Agent + Anti-Tracking enabled
⚠️ Click first search result untuk akurasi
⚠️ Auto-save tiap 50 bank



  0%|          | 1/2055 [00:28<16:30:06, 28.92s/it]

✓ [1] PT Bank Perekonomian Rakyat Cikarang Rah | (-6.258403, 107.133667)


  0%|          | 2/2055 [00:29<7:08:12, 12.51s/it] 

✓ [2] PT Bank Perekonomian Rakyat Dana Multi G | (-6.249312, 107.155654)


...

  1%|          | 20/2055 [03:23<4:45:03,  8.40s/it]

✓ [18] PT Bank Perekonomian Rakyat Usaha Rakyat | (-6.266879, 107.076519)


  1%|          | 21/2055 [03:34<5:10:27,  9.16s/it]

✓ [19] PT. BPR Sekar | (-6.338192, 107.119683)


  1%|          | 21/2055 [03:38<5:53:07, 10.42s/it]


In [6]:
all_banks.to_csv('BankLongLat.csv', index=False)